# 💼 Notebook commenté — Analyse des Offres d'Emploi LinkedIn avec Snowflake

## MBAESG — Big Data & Intelligence Artificielle

Ce notebook a été généré pour t'aider à **présenter et documenter le projet étape par étape**.

Il contient :
- le **contexte du projet** ;
- les **scripts SQL commentés** ;
- les **analyses à réaliser** ;
- la **partie Streamlit** ;
- des **explications pédagogiques** pour chaque bloc.

> ⚠️ **Important** : ce notebook est pensé comme un **support de travail/commentaire** et un **livrable pédagogique**.  
> Les scripts SQL doivent être exécutés dans **Snowflake Worksheets**.  
> Le code Streamlit doit être déployé dans **Snowflake Streamlit**.


## 1. Objectif du projet

Le projet consiste à analyser un ensemble de données LinkedIn stocké dans un bucket S3 public :

`s3://snowflake-lab-bucket/`

L'objectif est de :
1. créer une base de données **Snowflake** ;
2. charger les fichiers **CSV** et **JSON** ;
3. transformer les données JSON ;
4. produire **5 analyses métier** ;
5. visualiser les résultats avec **Streamlit**.

---

## 2. Jeux de données utilisés

Les principaux fichiers sont :
- `job_postings.csv`
- `benefits.csv`
- `employee_counts.csv`
- `job_skills.csv`
- `companies.json`
- `job_industries.json`
- `company_specialities.json`
- `company_industries.json`


## 3. Organisation recommandée du travail

L'ordre logique d'exécution est le suivant :

1. **Créer la base et les formats de fichiers**
2. **Créer les tables**
3. **Charger les données**
4. **Transformer les JSON en tables relationnelles**
5. **Exécuter les analyses**
6. **Construire le dashboard Streamlit**

Dans les sections ci-dessous, chaque étape est accompagnée de commentaires pour expliquer le rôle de chaque script.


## 4. Étape 1 — Setup Snowflake

Cette étape prépare l'environnement Snowflake :
- création de la base `linkedin` ;
- création du schéma `raw_data` ;
- création du **stage externe** pointant vers S3 ;
- création des **formats de fichiers** CSV et JSON.

### Pourquoi cette étape est importante ?
Sans elle, Snowflake ne sait ni **où lire les fichiers**, ni **comment les interpréter**.


```sql
-- ============================================================
-- SCRIPT 01 : SETUP DE LA BASE DE DONNÉES LINKEDIN
-- Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake
-- MBA Big Data & IA — MBAESG
-- ============================================================

-- ============================================================
-- 1. CRÉATION DE LA BASE DE DONNÉES ET DU SCHÉMA
-- ============================================================

-- Crée la base de données principale du projet
CREATE DATABASE IF NOT EXISTS linkedin;

-- On se place dans la base linkedin
USE DATABASE linkedin;

-- Crée le schéma raw_data qui contiendra toutes nos tables
CREATE SCHEMA IF NOT EXISTS raw_data;

-- On se place dans le schéma raw_data
USE SCHEMA raw_data;

-- ============================================================
-- 2. CRÉATION DU STAGE EXTERNE (S3)
-- Un "stage" est un pointeur vers une source de données externe.
-- Ici, il pointe vers le bucket S3 public fourni.
-- ============================================================

CREATE STAGE IF NOT EXISTS linkedin_stage
  URL = 's3://snowflake-lab-bucket/'
  COMMENT = 'Stage externe pointant vers le bucket S3 LinkedIn public';

-- Vérification : liste les fichiers disponibles dans le stage
LIST @linkedin_stage;

-- ============================================================
-- 3. DÉFINITION DES FORMATS DE FICHIERS
-- ============================================================

-- Format CSV :
-- - SKIP_HEADER = 1 : ignore la première ligne (en-tête)
-- - FIELD_OPTIONALLY_ENCLOSED_BY = '"' : gère les champs entre guillemets
-- - NULL_IF : traite ces valeurs comme NULL
-- - EMPTY_FIELD_AS_NULL : champ vide = NULL
-- - TRIM_SPACE : supprime les espaces autour des valeurs
CREATE OR REPLACE FILE FORMAT csv_format
  TYPE                      = 'CSV'
  FIELD_DELIMITER           = ','
  RECORD_DELIMITER          = '\n'
  SKIP_HEADER               = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF                   = ('NULL', 'null', '', 'N/A', 'n/a')
  EMPTY_FIELD_AS_NULL       = TRUE
  TRIM_SPACE                = TRUE
  COMMENT                   = 'Format pour fichiers CSV avec en-tête';

-- Format JSON :
-- - STRIP_OUTER_ARRAY = TRUE : retire le tableau racine [...] pour lire objet par objet
-- - IGNORE_UTF8_ERRORS : évite les erreurs d'encodage
CREATE OR REPLACE FILE FORMAT json_format
  TYPE               = 'JSON'
  STRIP_OUTER_ARRAY  = TRUE
  IGNORE_UTF8_ERRORS = TRUE
  COMMENT            = 'Format pour fichiers JSON avec tableau racine';

-- ============================================================
-- VÉRIFICATION : Affiche les formats créés
-- ============================================================
SHOW FILE FORMATS IN SCHEMA raw_data;

```


## 5. Étape 2 — Création des tables

Cette étape crée :
- les tables relationnelles pour les fichiers CSV ;
- les tables **RAW** de type `VARIANT` pour les fichiers JSON ;
- les tables finales qui recevront les données JSON transformées.

### Point pédagogique
Dans Snowflake, il est très courant de charger le JSON brut dans une colonne `VARIANT`, puis d'extraire les champs utiles avec la syntaxe :

```sql
raw_data:nom_du_champ::TYPE
```


```sql
-- ============================================================
-- SCRIPT 02 : CRÉATION DES TABLES
-- Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake
-- ============================================================

USE DATABASE linkedin;
USE SCHEMA raw_data;

-- ============================================================
-- TABLE 1 : job_postings (source : job_postings.csv)
-- Table principale contenant toutes les offres d'emploi
-- ============================================================
CREATE OR REPLACE TABLE job_postings (
    job_id                     VARCHAR(50)    COMMENT 'Identifiant unique LinkedIn de l offre',
    company_name               VARCHAR(255)   COMMENT 'Nom de l entreprise',
    title                      VARCHAR(255)   COMMENT 'Titre du poste',
    description                TEXT           COMMENT 'Description complète du poste',
    max_salary                 FLOAT          COMMENT 'Salaire maximum proposé',
    med_salary                 FLOAT          COMMENT 'Salaire médian proposé',
    min_salary                 FLOAT          COMMENT 'Salaire minimum proposé',
    pay_period                 VARCHAR(50)    COMMENT 'Périodicité du salaire (HOURLY, MONTHLY, YEARLY)',
    formatted_work_type        VARCHAR(100)   COMMENT 'Type de travail formaté (Full-time, Part-time, Contract...)',
    location                   VARCHAR(255)   COMMENT 'Localisation du poste',
    applies                    INTEGER        COMMENT 'Nombre de candidatures reçues',
    original_listed_time       BIGINT         COMMENT 'Timestamp Unix de la publication initiale',
    remote_allowed             BOOLEAN        COMMENT 'TRUE si le télétravail est autorisé',
    views                      INTEGER        COMMENT 'Nombre de vues de l offre',
    job_posting_url            VARCHAR(500)   COMMENT 'URL de l offre sur LinkedIn',
    application_url            VARCHAR(500)   COMMENT 'URL pour postuler',
    application_type           VARCHAR(100)   COMMENT 'Type de candidature (offsite, onsite simple/complexe)',
    expiry                     BIGINT         COMMENT 'Timestamp Unix d expiration de l offre',
    closed_time                BIGINT         COMMENT 'Timestamp Unix de fermeture',
    formatted_experience_level VARCHAR(100)   COMMENT 'Niveau d expérience (Entry, Associate, Mid-Senior...)',
    skills_desc                TEXT           COMMENT 'Description des compétences requises',
    listed_time                BIGINT         COMMENT 'Timestamp Unix de la mise en ligne',
    posting_domain             VARCHAR(255)   COMMENT 'Domaine du site de candidature',
    sponsored                  BOOLEAN        COMMENT 'TRUE si l offre est sponsorisée',
    work_type                  VARCHAR(100)   COMMENT 'Type de travail brut',
    currency                   VARCHAR(10)    COMMENT 'Devise du salaire (USD, EUR...)',
    compensation_type          VARCHAR(100)   COMMENT 'Type de rémunération'
);

-- ============================================================
-- TABLE 2 : benefits (source : benefits.csv)
-- Avantages liés à chaque offre d'emploi
-- ============================================================
CREATE OR REPLACE TABLE benefits (
    job_id    VARCHAR(50)  COMMENT 'Référence à job_postings.job_id',
    inferred  BOOLEAN      COMMENT 'TRUE si l avantage a été inféré par LinkedIn (non déclaré)',
    type      VARCHAR(100) COMMENT 'Type d avantage (401K, Medical Insurance, Dental...)'
);

-- ============================================================
-- TABLE 3 : employee_counts (source : employee_counts.csv)
-- Statistiques RH des entreprises
-- ============================================================
CREATE OR REPLACE TABLE employee_counts (
    company_id      VARCHAR(50) COMMENT 'Identifiant de l entreprise',
    employee_count  INTEGER     COMMENT 'Nombre d employés',
    follower_count  INTEGER     COMMENT 'Nombre de followers LinkedIn',
    time_recorded   BIGINT      COMMENT 'Timestamp Unix de la collecte des données'
);

-- ============================================================
-- TABLE 4 : job_skills (source : job_skills.csv)
-- Compétences associées aux offres d'emploi
-- ============================================================
CREATE OR REPLACE TABLE job_skills (
    job_id     VARCHAR(50) COMMENT 'Référence à job_postings.job_id',
    skill_abr  VARCHAR(50) COMMENT 'Abréviation de la compétence requise'
);

-- ============================================================
-- TABLES INTERMÉDIAIRES (VARIANT) POUR LES FICHIERS JSON
-- Les fichiers JSON sont d'abord chargés sous forme brute (VARIANT),
-- puis transformés en tables relationnelles (Étape 4).
-- ============================================================

-- Table brute pour companies.json
CREATE OR REPLACE TABLE companies_raw (
    raw_data VARIANT COMMENT 'Données brutes JSON pour companies'
);

-- Table brute pour job_industries.json
CREATE OR REPLACE TABLE job_industries_raw (
    raw_data VARIANT COMMENT 'Données brutes JSON pour job_industries'
);

-- Table brute pour company_specialities.json
CREATE OR REPLACE TABLE company_specialities_raw (
    raw_data VARIANT COMMENT 'Données brutes JSON pour company_specialities'
);

-- Table brute pour company_industries.json
CREATE OR REPLACE TABLE company_industries_raw (
    raw_data VARIANT COMMENT 'Données brutes JSON pour company_industries'
);

-- ============================================================
-- TABLES FINALES POUR LES DONNÉES JSON (après transformation)
-- ============================================================

-- TABLE 5 : companies (source : companies.json)
CREATE OR REPLACE TABLE companies (
    company_id    VARCHAR(50)  COMMENT 'Identifiant LinkedIn de l entreprise',
    name          VARCHAR(255) COMMENT 'Nom de l entreprise',
    description   TEXT         COMMENT 'Description de l entreprise',
    company_size  INTEGER      COMMENT 'Taille (0=<10 employés ... 7=>10000)',
    state         VARCHAR(100) COMMENT 'État du siège social',
    country       VARCHAR(100) COMMENT 'Pays du siège social',
    city          VARCHAR(100) COMMENT 'Ville du siège social',
    zip_code      VARCHAR(20)  COMMENT 'Code postal',
    address       VARCHAR(500) COMMENT 'Adresse complète',
    url           VARCHAR(500) COMMENT 'URL de la page LinkedIn de l entreprise'
);

-- TABLE 6 : job_industries (source : job_industries.json)
CREATE OR REPLACE TABLE job_industries (
    job_id       VARCHAR(50) COMMENT 'Référence à job_postings.job_id',
    industry_id  VARCHAR(50) COMMENT 'Identifiant du secteur d activité'
);

-- TABLE 7 : company_specialities (source : company_specialities.json)
CREATE OR REPLACE TABLE company_specialities (
    company_id  VARCHAR(50)  COMMENT 'Référence à companies.company_id',
    speciality  VARCHAR(255) COMMENT 'Spécialité de l entreprise'
);

-- TABLE 8 : company_industries (source : company_industries.json)
CREATE OR REPLACE TABLE company_industries (
    company_id  VARCHAR(50)  COMMENT 'Référence à companies.company_id',
    industry    VARCHAR(255) COMMENT 'Secteur d activité de l entreprise'
);

-- ============================================================
-- VÉRIFICATION : Liste toutes les tables créées
-- ============================================================
SHOW TABLES IN SCHEMA raw_data;

```


## 6. Étape 3 — Chargement des données avec `COPY INTO`

Ici, on importe les données depuis le bucket S3 vers Snowflake.

### Pourquoi utiliser `COPY INTO` ?
`COPY INTO` est la commande standard Snowflake pour **ingérer des données** depuis un stage.

### Remarque importante
L'option :

```sql
ON_ERROR = 'CONTINUE'
```

permet de **ne pas bloquer tout le chargement** si quelques lignes sont mal formées.


```sql
-- ============================================================
-- SCRIPT 03 : CHARGEMENT DES DONNÉES (COPY INTO)
-- Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake
-- ============================================================
-- Ce script importe les données depuis le bucket S3 vers Snowflake.
-- ON_ERROR = 'CONTINUE' : ignore les lignes mal formées sans bloquer.
-- ============================================================

USE DATABASE linkedin;
USE SCHEMA raw_data;

-- ============================================================
-- CHARGEMENT DES FICHIERS CSV
-- ============================================================

-- 1. Chargement de job_postings.csv
-- Fichier principal : toutes les offres d'emploi LinkedIn
COPY INTO job_postings
FROM @linkedin_stage/job_postings.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR    = 'CONTINUE';

-- Vérification rapide
SELECT COUNT(*) AS nb_offres_chargees FROM job_postings;

-- -------------------------------------------------------

-- 2. Chargement de benefits.csv
-- Avantages par offre d'emploi
COPY INTO benefits
FROM @linkedin_stage/benefits.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_avantages_charges FROM benefits;

-- -------------------------------------------------------

-- 3. Chargement de employee_counts.csv
-- Nombre d'employés et de followers par entreprise
COPY INTO employee_counts
FROM @linkedin_stage/employee_counts.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_entreprises_rh FROM employee_counts;

-- -------------------------------------------------------

-- 4. Chargement de job_skills.csv
-- Compétences requises par offre d'emploi
COPY INTO job_skills
FROM @linkedin_stage/job_skills.csv
FILE_FORMAT = (FORMAT_NAME = 'csv_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_competences_chargees FROM job_skills;

-- ============================================================
-- CHARGEMENT DES FICHIERS JSON (dans les tables RAW/VARIANT)
-- Les fichiers JSON sont chargés dans des tables intermédiaires
-- puis transformés en tables relationnelles dans le script 04.
-- ============================================================

-- 5. Chargement de companies.json
-- Informations détaillées sur chaque entreprise
COPY INTO companies_raw
FROM @linkedin_stage/companies.json
FILE_FORMAT = (FORMAT_NAME = 'json_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_entreprises_raw FROM companies_raw;

-- -------------------------------------------------------

-- 6. Chargement de job_industries.json
-- Secteurs d'activité par offre d'emploi
COPY INTO job_industries_raw
FROM @linkedin_stage/job_industries.json
FILE_FORMAT = (FORMAT_NAME = 'json_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_job_industries_raw FROM job_industries_raw;

-- -------------------------------------------------------

-- 7. Chargement de company_specialities.json
-- Spécialités par entreprise
COPY INTO company_specialities_raw
FROM @linkedin_stage/company_specialities.json
FILE_FORMAT = (FORMAT_NAME = 'json_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_specialites_raw FROM company_specialities_raw;

-- -------------------------------------------------------

-- 8. Chargement de company_industries.json
-- Secteurs d'activité par entreprise
COPY INTO company_industries_raw
FROM @linkedin_stage/company_industries.json
FILE_FORMAT = (FORMAT_NAME = 'json_format')
ON_ERROR    = 'CONTINUE';

SELECT COUNT(*) AS nb_industries_raw FROM company_industries_raw;

-- ============================================================
-- VÉRIFICATION GLOBALE DU CHARGEMENT
-- Affiche le nombre de lignes chargées pour chaque table brute
-- ============================================================
SELECT 'job_postings'            AS table_name, COUNT(*) AS nb_lignes FROM job_postings
UNION ALL
SELECT 'benefits',                COUNT(*) FROM benefits
UNION ALL
SELECT 'employee_counts',         COUNT(*) FROM employee_counts
UNION ALL
SELECT 'job_skills',              COUNT(*) FROM job_skills
UNION ALL
SELECT 'companies_raw',           COUNT(*) FROM companies_raw
UNION ALL
SELECT 'job_industries_raw',      COUNT(*) FROM job_industries_raw
UNION ALL
SELECT 'company_specialities_raw',COUNT(*) FROM company_specialities_raw
UNION ALL
SELECT 'company_industries_raw',  COUNT(*) FROM company_industries_raw
ORDER BY table_name;

```


## 7. Étape 4 — Transformation des fichiers JSON

Les fichiers JSON ne sont pas directement exploitables pour les analyses métier si on les laisse sous forme brute.

On les transforme donc en tables relationnelles, par exemple :
- `companies_raw` → `companies`
- `job_industries_raw` → `job_industries`
- `company_specialities_raw` → `company_specialities`
- `company_industries_raw` → `company_industries`

### Exemple de transformation
```sql
raw_data:company_id::VARCHAR(50)
```
Cela signifie :
- prendre le champ `company_id` dans le JSON ;
- le convertir en texte (`VARCHAR`) ;
- l'insérer dans la table finale.


```sql
-- ============================================================
-- SCRIPT 04 : TRANSFORMATIONS JSON → TABLES RELATIONNELLES
-- Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake
-- ============================================================
-- Ce script extrait les champs des colonnes VARIANT (JSON brut)
-- et les insère dans les tables relationnelles structurées.
-- Syntaxe Snowflake : raw_data:nom_du_champ::TYPE_DE_DONNÉE
-- ============================================================

USE DATABASE linkedin;
USE SCHEMA raw_data;

-- ============================================================
-- TRANSFORMATION 1 : companies_raw → companies
-- ============================================================
-- On extrait chaque champ JSON et on le convertit au bon type SQL
INSERT INTO companies
SELECT
    raw_data:company_id::VARCHAR(50)   AS company_id,
    raw_data:name::VARCHAR(255)        AS name,
    raw_data:description::TEXT         AS description,
    raw_data:company_size::INTEGER     AS company_size,
    raw_data:state::VARCHAR(100)       AS state,
    raw_data:country::VARCHAR(100)     AS country,
    raw_data:city::VARCHAR(100)        AS city,
    raw_data:zip_code::VARCHAR(20)     AS zip_code,
    raw_data:address::VARCHAR(500)     AS address,
    raw_data:url::VARCHAR(500)         AS url
FROM companies_raw;

-- Vérification
SELECT COUNT(*) AS nb_companies FROM companies;
SELECT * FROM companies LIMIT 5;

-- ============================================================
-- TRANSFORMATION 2 : job_industries_raw → job_industries
-- ============================================================
INSERT INTO job_industries
SELECT
    raw_data:job_id::VARCHAR(50)       AS job_id,
    raw_data:industry_id::VARCHAR(50)  AS industry_id
FROM job_industries_raw;

-- Vérification
SELECT COUNT(*) AS nb_job_industries FROM job_industries;
SELECT * FROM job_industries LIMIT 5;

-- ============================================================
-- TRANSFORMATION 3 : company_specialities_raw → company_specialities
-- ============================================================
INSERT INTO company_specialities
SELECT
    raw_data:company_id::VARCHAR(50)   AS company_id,
    raw_data:speciality::VARCHAR(255)  AS speciality
FROM company_specialities_raw;

-- Vérification
SELECT COUNT(*) AS nb_specialities FROM company_specialities;
SELECT * FROM company_specialities LIMIT 5;

-- ============================================================
-- TRANSFORMATION 4 : company_industries_raw → company_industries
-- ============================================================
INSERT INTO company_industries
SELECT
    raw_data:company_id::VARCHAR(50)   AS company_id,
    raw_data:industry::VARCHAR(255)    AS industry
FROM company_industries_raw;

-- Vérification
SELECT COUNT(*) AS nb_company_industries FROM company_industries;
SELECT * FROM company_industries LIMIT 5;

-- ============================================================
-- VÉRIFICATION GLOBALE FINALE
-- Toutes les tables doivent avoir des données non nulles
-- ============================================================
SELECT 'job_postings'          AS table_name, COUNT(*) AS nb_lignes FROM job_postings
UNION ALL
SELECT 'benefits',              COUNT(*) FROM benefits
UNION ALL
SELECT 'employee_counts',       COUNT(*) FROM employee_counts
UNION ALL
SELECT 'job_skills',            COUNT(*) FROM job_skills
UNION ALL
SELECT 'companies',             COUNT(*) FROM companies
UNION ALL
SELECT 'job_industries',        COUNT(*) FROM job_industries
UNION ALL
SELECT 'company_specialities',  COUNT(*) FROM company_specialities
UNION ALL
SELECT 'company_industries',    COUNT(*) FROM company_industries
ORDER BY nb_lignes DESC;

-- ============================================================
-- CONTRÔLE QUALITÉ DES DONNÉES
-- ============================================================

-- Vérifie les valeurs NULL dans les colonnes clés de job_postings
SELECT
    COUNT(*)                                      AS total_offres,
    COUNT(job_id)                                 AS avec_job_id,
    COUNT(title)                                  AS avec_titre,
    COUNT(company_name)                           AS avec_entreprise,
    COUNT(med_salary)                             AS avec_salaire_median,
    COUNT(formatted_work_type)                    AS avec_type_emploi,
    ROUND(COUNT(med_salary) * 100.0 / COUNT(*), 1) AS pct_offres_avec_salaire
FROM job_postings;

-- Vérifie les types d'emploi distincts
SELECT formatted_work_type, COUNT(*) AS nb
FROM job_postings
GROUP BY formatted_work_type
ORDER BY nb DESC;

-- Vérifie la distribution des tailles d'entreprise
SELECT company_size, COUNT(*) AS nb_entreprises
FROM companies
GROUP BY company_size
ORDER BY company_size;

```


## 8. Étape 5 — Analyses SQL

Le sujet demande 5 analyses principales :

1. **Top 10 des titres de postes les plus publiés par industrie**
2. **Top 10 des postes les mieux rémunérés par industrie**
3. **Répartition des offres par taille d'entreprise**
4. **Répartition des offres par secteur d'activité**
5. **Répartition des offres par type d'emploi**

### Concepts SQL mobilisés
Dans ces requêtes, on utilise notamment :
- des **jointures** ;
- des **CTE** (`WITH ... AS`) ;
- des **fonctions analytiques** comme `ROW_NUMBER()` ;
- des calculs de **pourcentage** avec `SUM() OVER()`.


```sql
-- ============================================================
-- SCRIPT 05 : ANALYSES DES DONNÉES
-- Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake
-- ============================================================

USE DATABASE linkedin;
USE SCHEMA raw_data;

-- ============================================================
-- ANALYSE 1 : Top 10 des titres de postes les plus publiés
--             par industrie
-- ============================================================
-- Objectif : Identifier les postes les plus demandés dans
--            chaque secteur d'activité.
-- Technique : CTE + ROW_NUMBER() partitionné par industrie
-- ============================================================
WITH ranked_jobs AS (
    SELECT
        ci.industry                   AS industrie,
        jp.title                      AS titre_poste,
        COUNT(*)                      AS nb_offres,
        ROW_NUMBER() OVER (
            PARTITION BY ci.industry
            ORDER BY COUNT(*) DESC
        )                             AS rang
    FROM job_postings  jp
    JOIN job_industries     ji ON jp.job_id      = ji.job_id
    JOIN company_industries ci ON ji.industry_id = ci.industry
    GROUP BY ci.industry, jp.title
)
SELECT
    industrie,
    rang,
    titre_poste,
    nb_offres
FROM ranked_jobs
WHERE rang <= 10
ORDER BY industrie, rang;

-- ============================================================
-- ANALYSE 2 : Top 10 des postes les mieux rémunérés
--             par industrie
-- ============================================================
-- Objectif : Identifier les postes les plus rémunérateurs
--            dans chaque secteur.
-- Note : Basé sur le salaire médian moyen (med_salary).
--        Les offres sans salaire sont exclues (WHERE IS NOT NULL).
-- ============================================================
WITH salary_ranked AS (
    SELECT
        ci.industry                          AS industrie,
        jp.title                             AS titre_poste,
        ROUND(AVG(jp.med_salary), 0)         AS salaire_median_moyen,
        COUNT(*)                             AS nb_offres,
        ROW_NUMBER() OVER (
            PARTITION BY ci.industry
            ORDER BY AVG(jp.med_salary) DESC NULLS LAST
        )                                    AS rang
    FROM job_postings  jp
    JOIN job_industries     ji ON jp.job_id      = ji.job_id
    JOIN company_industries ci ON ji.industry_id = ci.industry
    WHERE jp.med_salary IS NOT NULL
    GROUP BY ci.industry, jp.title
)
SELECT
    industrie,
    rang,
    titre_poste,
    salaire_median_moyen,
    nb_offres
FROM salary_ranked
WHERE rang <= 10
ORDER BY industrie, rang;

-- ============================================================
-- ANALYSE 3 : Répartition des offres d'emploi
--             par taille d'entreprise
-- ============================================================
-- Objectif : Savoir si les grandes ou les petites entreprises
--            publient le plus d'offres sur LinkedIn.
-- Note : company_size va de 0 (très petite) à 7 (multinationale)
-- ============================================================
SELECT
    CASE c.company_size
        WHEN 0 THEN '0 — Très petite   (< 10 emp.)'
        WHEN 1 THEN '1 — Petite        (10–50 emp.)'
        WHEN 2 THEN '2 — PME           (51–200 emp.)'
        WHEN 3 THEN '3 — Moyenne       (201–500 emp.)'
        WHEN 4 THEN '4 — Grande        (501–1 000 emp.)'
        WHEN 5 THEN '5 — Très grande   (1 001–5 000 emp.)'
        WHEN 6 THEN '6 — Entreprise    (5 001–10 000 emp.)'
        WHEN 7 THEN '7 — Multinationale(> 10 000 emp.)'
        ELSE        'Non renseignée'
    END                                                          AS taille_entreprise,
    COUNT(jp.job_id)                                             AS nb_offres,
    ROUND(COUNT(jp.job_id) * 100.0 / SUM(COUNT(jp.job_id)) OVER(), 2) AS pourcentage
FROM job_postings jp
LEFT JOIN companies c
       ON TRIM(LOWER(jp.company_name)) = TRIM(LOWER(c.name))
GROUP BY c.company_size
ORDER BY c.company_size NULLS LAST;

-- ============================================================
-- ANALYSE 4 : Répartition des offres par secteur d'activité
-- ============================================================
-- Objectif : Identifier les secteurs les plus actifs
--            en recrutement sur LinkedIn.
-- ============================================================
SELECT
    ci.industry                                                  AS secteur_activite,
    COUNT(DISTINCT jp.job_id)                                    AS nb_offres,
    ROUND(
        COUNT(DISTINCT jp.job_id) * 100.0
        / SUM(COUNT(DISTINCT jp.job_id)) OVER(),
        2
    )                                                            AS pourcentage
FROM job_postings  jp
JOIN job_industries     ji ON jp.job_id      = ji.job_id
JOIN company_industries ci ON ji.industry_id = ci.industry
GROUP BY ci.industry
ORDER BY nb_offres DESC
LIMIT 20;

-- ============================================================
-- ANALYSE 5 : Répartition des offres par type d'emploi
-- ============================================================
-- Objectif : Connaître la répartition CDI / CDD /
--            Temps partiel / Stage / Contrat sur LinkedIn.
-- ============================================================
SELECT
    COALESCE(formatted_work_type, 'Non spécifié')               AS type_emploi,
    COUNT(*)                                                     AS nb_offres,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2)           AS pourcentage
FROM job_postings
GROUP BY formatted_work_type
ORDER BY nb_offres DESC;

```


## 9. Interprétation attendue des analyses

Voici ce que tu peux expliquer dans ton rendu :

### Analyse 1 — Postes les plus publiés
Elle permet d'identifier les **métiers les plus demandés** dans chaque secteur.

### Analyse 2 — Postes les mieux rémunérés
Elle met en évidence les **rôles les plus valorisés financièrement**.

### Analyse 3 — Taille d'entreprise
Elle permet de voir si ce sont plutôt les **petites**, **moyennes** ou **grandes entreprises** qui recrutent le plus.

### Analyse 4 — Secteurs d'activité
Elle donne une vision globale des **secteurs les plus dynamiques** sur LinkedIn.

### Analyse 5 — Type d'emploi
Elle montre la structure du marché : **temps plein**, **temps partiel**, **contrat**, **stage**, etc.


## 10. Partie Streamlit

Une fois les analyses validées dans Snowflake, on peut créer une application **Streamlit** pour les visualiser.

### Ce que fait l'application
- affiche des **KPIs globaux** ;
- propose **5 onglets** correspondant aux 5 analyses ;
- utilise des graphiques **Altair** ;
- permet une lecture plus claire et plus professionnelle des résultats.

### Déploiement
Dans Snowflake :

`Projects > Streamlit > + Streamlit App`

Puis il faut coller le code ci-dessous.


```python
# ============================================================
# APPLICATION STREAMLIT — LinkedIn Jobs Analysis Dashboard
# Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake
# MBA Big Data & IA — MBAESG
# Fichier : streamlit_app.py
#
# ⚠️ DÉPLOIEMENT : Cette app doit être déployée directement
#    dans Snowflake via : Projects > Streamlit > + Streamlit App
#    Elle utilise get_active_session() pour se connecter
#    automatiquement à la base linkedin.
# ============================================================

import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

# ============================================================
# CONFIGURATION DE LA PAGE
# ============================================================
st.set_page_config(
    page_title="LinkedIn Jobs Dashboard",
    page_icon="💼",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# ============================================================
# CSS PERSONNALISÉ — Thème LinkedIn
# ============================================================
st.markdown("""
<style>
    /* Fond principal */
    .main { background-color: #F3F2EF; }

    /* Titre principal */
    .main-title {
        font-size: 2.4rem;
        font-weight: 800;
        color: #0077B5;
        text-align: center;
        padding: 10px 0 0 0;
    }
    .sub-title {
        font-size: 1.1rem;
        color: #666;
        text-align: center;
        margin-bottom: 20px;
    }

    /* Cartes métriques */
    [data-testid="metric-container"] {
        background-color: white;
        border: 1px solid #e0e0e0;
        border-radius: 12px;
        padding: 15px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.07);
    }

    /* Onglets */
    .stTabs [data-baseweb="tab-list"] {
        gap: 8px;
    }
    .stTabs [data-baseweb="tab"] {
        background-color: white;
        border-radius: 8px 8px 0 0;
        border: 1px solid #ddd;
        font-weight: 600;
    }

    /* Séparateur */
    hr { border-color: #0077B5; opacity: 0.3; }
</style>
""", unsafe_allow_html=True)

# ============================================================
# CONNEXION SNOWFLAKE (automatique dans Snowflake Streamlit)
# ============================================================
session = get_active_session()

# ============================================================
# EN-TÊTE DU DASHBOARD
# ============================================================
st.markdown('<p class="main-title">💼 LinkedIn Jobs Market Dashboard</p>', unsafe_allow_html=True)
st.markdown('<p class="sub-title">Analyse des offres d\'emploi LinkedIn • MBA Big Data & IA — MBAESG</p>', unsafe_allow_html=True)
st.markdown("---")

# ============================================================
# MÉTRIQUES GLOBALES (KPIs en haut de page)
# ============================================================
@st.cache_data
def get_kpis():
    total_jobs      = session.sql("SELECT COUNT(*) FROM linkedin.raw_data.job_postings").collect()[0][0]
    total_companies = session.sql("SELECT COUNT(DISTINCT company_name) FROM linkedin.raw_data.job_postings").collect()[0][0]
    total_industries= session.sql("SELECT COUNT(DISTINCT industry) FROM linkedin.raw_data.company_industries").collect()[0][0]
    avg_salary_row  = session.sql(
        "SELECT ROUND(AVG(med_salary),0) FROM linkedin.raw_data.job_postings WHERE med_salary IS NOT NULL"
    ).collect()[0][0]
    remote_pct      = session.sql(
        "SELECT ROUND(COUNT(*) FILTER(WHERE remote_allowed=TRUE) * 100.0 / COUNT(*), 1) FROM linkedin.raw_data.job_postings"
    ).collect()[0][0]
    return total_jobs, total_companies, total_industries, avg_salary_row, remote_pct

total_jobs, total_companies, total_industries, avg_salary, remote_pct = get_kpis()

col1, col2, col3, col4, col5 = st.columns(5)
with col1:
    st.metric("📋 Total Offres",       f"{total_jobs:,}")
with col2:
    st.metric("🏢 Entreprises",        f"{total_companies:,}")
with col3:
    st.metric("🏭 Secteurs d'activité",f"{total_industries:,}")
with col4:
    st.metric("💰 Salaire Médian Moy.",f"${avg_salary:,.0f}" if avg_salary else "N/A")
with col5:
    st.metric("🏠 Offres Remote",      f"{remote_pct}%" if remote_pct else "N/A")

st.markdown("---")

# ============================================================
# ONGLETS DES 5 ANALYSES
# ============================================================
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "🏆 Top 10 Titres / Industrie",
    "💰 Top 10 Salaires / Industrie",
    "🏢 Taille d'Entreprise",
    "🏭 Secteurs d'Activité",
    "⏱️ Types d'Emploi"
])

# ============================================================
# TAB 1 — Top 10 titres de postes par industrie
# ============================================================
with tab1:
    st.subheader("🏆 Top 10 des titres de postes les plus publiés par industrie")
    st.caption("Utilisez le menu déroulant pour explorer chaque secteur d'activité.")

    @st.cache_data
    def get_top_titles():
        query = """
        WITH ranked_jobs AS (
            SELECT
                ci.industry   AS INDUSTRIE,
                jp.title      AS TITRE_POSTE,
                COUNT(*)      AS NB_OFFRES,
                ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY COUNT(*) DESC) AS RANG
            FROM linkedin.raw_data.job_postings jp
            JOIN linkedin.raw_data.job_industries     ji ON jp.job_id      = ji.job_id
            JOIN linkedin.raw_data.company_industries ci ON ji.industry_id = ci.industry
            GROUP BY ci.industry, jp.title
        )
        SELECT INDUSTRIE, TITRE_POSTE, NB_OFFRES, RANG
        FROM ranked_jobs
        WHERE RANG <= 10
        ORDER BY INDUSTRIE, RANG
        """
        return session.sql(query).to_pandas()

    df1 = get_top_titles()
    industries1 = sorted(df1['INDUSTRIE'].dropna().unique())

    selected_ind1 = st.selectbox(
        "🔍 Sélectionnez un secteur d'activité :",
        options=industries1,
        key="select_tab1"
    )

    df1_filtered = df1[df1['INDUSTRIE'] == selected_ind1].sort_values('NB_OFFRES', ascending=False)

    # Graphique barres horizontales
    chart1 = alt.Chart(df1_filtered).mark_bar(
        color='#0077B5',
        cornerRadiusTopRight=5,
        cornerRadiusBottomRight=5
    ).encode(
        x=alt.X('NB_OFFRES:Q', title="Nombre d'offres"),
        y=alt.Y('TITRE_POSTE:N', sort='-x', title=""),
        tooltip=[
            alt.Tooltip('TITRE_POSTE:N', title='Poste'),
            alt.Tooltip('NB_OFFRES:Q',   title="Nb d'offres")
        ]
    ).properties(
        title=f"Top 10 postes — {selected_ind1}",
        height=400
    )

    # Annotation du nombre sur chaque barre
    text1 = chart1.mark_text(align='left', dx=5, color='#333').encode(
        text=alt.Text('NB_OFFRES:Q')
    )

    st.altair_chart(chart1 + text1, use_container_width=True)

    with st.expander("📋 Voir le tableau de données"):
        st.dataframe(
            df1_filtered[['RANG', 'TITRE_POSTE', 'NB_OFFRES']].reset_index(drop=True),
            use_container_width=True
        )

# ============================================================
# TAB 2 — Top 10 postes les mieux rémunérés par industrie
# ============================================================
with tab2:
    st.subheader("💰 Top 10 des postes les mieux rémunérés par industrie")
    st.caption("Basé sur le salaire médian moyen annuel. Seules les offres avec salaire renseigné sont incluses.")

    @st.cache_data
    def get_top_salaries():
        query = """
        WITH salary_ranked AS (
            SELECT
                ci.industry                      AS INDUSTRIE,
                jp.title                         AS TITRE_POSTE,
                ROUND(AVG(jp.med_salary), 0)     AS SALAIRE_MEDIAN,
                COUNT(*)                         AS NB_OFFRES,
                ROW_NUMBER() OVER (
                    PARTITION BY ci.industry
                    ORDER BY AVG(jp.med_salary) DESC NULLS LAST
                )                                AS RANG
            FROM linkedin.raw_data.job_postings jp
            JOIN linkedin.raw_data.job_industries     ji ON jp.job_id      = ji.job_id
            JOIN linkedin.raw_data.company_industries ci ON ji.industry_id = ci.industry
            WHERE jp.med_salary IS NOT NULL
            GROUP BY ci.industry, jp.title
        )
        SELECT INDUSTRIE, TITRE_POSTE, SALAIRE_MEDIAN, NB_OFFRES, RANG
        FROM salary_ranked
        WHERE RANG <= 10
        ORDER BY INDUSTRIE, RANG
        """
        return session.sql(query).to_pandas()

    df2 = get_top_salaries()
    industries2 = sorted(df2['INDUSTRIE'].dropna().unique())

    selected_ind2 = st.selectbox(
        "🔍 Sélectionnez un secteur d'activité :",
        options=industries2,
        key="select_tab2"
    )

    df2_filtered = df2[df2['INDUSTRIE'] == selected_ind2].sort_values('SALAIRE_MEDIAN', ascending=False)

    chart2 = alt.Chart(df2_filtered).mark_bar(
        color='#00A0DC',
        cornerRadiusTopRight=5,
        cornerRadiusBottomRight=5
    ).encode(
        x=alt.X('SALAIRE_MEDIAN:Q',
                title="Salaire médian ($)",
                axis=alt.Axis(format='$,.0f')),
        y=alt.Y('TITRE_POSTE:N', sort='-x', title=""),
        tooltip=[
            alt.Tooltip('TITRE_POSTE:N',   title='Poste'),
            alt.Tooltip('SALAIRE_MEDIAN:Q', title='Salaire médian ($)', format='$,.0f'),
            alt.Tooltip('NB_OFFRES:Q',      title="Nb d'offres")
        ]
    ).properties(
        title=f"Top 10 salaires — {selected_ind2}",
        height=400
    )

    text2 = chart2.mark_text(align='left', dx=5, color='#333').encode(
        text=alt.Text('SALAIRE_MEDIAN:Q', format='$,.0f')
    )

    st.altair_chart(chart2 + text2, use_container_width=True)

    with st.expander("📋 Voir le tableau de données"):
        df2_display = df2_filtered.copy()
        df2_display['SALAIRE_MEDIAN'] = df2_display['SALAIRE_MEDIAN'].apply(lambda x: f"${x:,.0f}")
        st.dataframe(
            df2_display[['RANG', 'TITRE_POSTE', 'SALAIRE_MEDIAN', 'NB_OFFRES']].reset_index(drop=True),
            use_container_width=True
        )

# ============================================================
# TAB 3 — Répartition par taille d'entreprise
# ============================================================
with tab3:
    st.subheader("🏢 Répartition des offres d'emploi par taille d'entreprise")
    st.caption("La taille varie de 0 (< 10 employés) à 7 (> 10 000 employés).")

    @st.cache_data
    def get_by_company_size():
        query = """
        SELECT
            CASE c.company_size
                WHEN 0 THEN '0 — Très petite (< 10)'
                WHEN 1 THEN '1 — Petite (10–50)'
                WHEN 2 THEN '2 — PME (51–200)'
                WHEN 3 THEN '3 — Moyenne (201–500)'
                WHEN 4 THEN '4 — Grande (501–1000)'
                WHEN 5 THEN '5 — Très grande (1001–5000)'
                WHEN 6 THEN '6 — Entreprise (5001–10000)'
                WHEN 7 THEN '7 — Multinationale (> 10000)'
                ELSE 'Non renseignée'
            END                AS TAILLE_ENTREPRISE,
            COUNT(jp.job_id)   AS NB_OFFRES
        FROM linkedin.raw_data.job_postings jp
        LEFT JOIN linkedin.raw_data.companies c
               ON TRIM(LOWER(jp.company_name)) = TRIM(LOWER(c.name))
        GROUP BY c.company_size
        ORDER BY c.company_size NULLS LAST
        """
        return session.sql(query).to_pandas()

    df3 = get_by_company_size()
    total3 = df3['NB_OFFRES'].sum()
    df3['POURCENTAGE'] = (df3['NB_OFFRES'] / total3 * 100).round(1)

    col_bar3, col_pie3 = st.columns([3, 2])

    with col_bar3:
        chart3_bar = alt.Chart(df3).mark_bar(color='#0077B5', cornerRadiusTopRight=5).encode(
            x=alt.X('TAILLE_ENTREPRISE:N', sort=None, title="Taille d'entreprise",
                    axis=alt.Axis(labelAngle=-25, labelFontSize=11)),
            y=alt.Y('NB_OFFRES:Q', title="Nombre d'offres"),
            tooltip=[
                alt.Tooltip('TAILLE_ENTREPRISE:N', title='Taille'),
                alt.Tooltip('NB_OFFRES:Q', title="Nb d'offres"),
                alt.Tooltip('POURCENTAGE:Q', title='%', format='.1f')
            ]
        ).properties(title="Offres par taille d'entreprise", height=380)
        st.altair_chart(chart3_bar, use_container_width=True)

    with col_pie3:
        chart3_donut = alt.Chart(df3).mark_arc(innerRadius=60, outerRadius=130).encode(
            theta=alt.Theta('NB_OFFRES:Q'),
            color=alt.Color('TAILLE_ENTREPRISE:N',
                           scale=alt.Scale(scheme='blues'),
                           legend=alt.Legend(title="Taille", orient='bottom', columns=2)),
            tooltip=[
                alt.Tooltip('TAILLE_ENTREPRISE:N', title='Taille'),
                alt.Tooltip('NB_OFFRES:Q',          title="Nb d'offres"),
                alt.Tooltip('POURCENTAGE:Q',         title='%', format='.1f')
            ]
        ).properties(title="Répartition en %", height=380)
        st.altair_chart(chart3_donut, use_container_width=True)

    with st.expander("📋 Voir le tableau de données"):
        df3['POURCENTAGE'] = df3['POURCENTAGE'].astype(str) + '%'
        st.dataframe(df3[['TAILLE_ENTREPRISE','NB_OFFRES','POURCENTAGE']], use_container_width=True)

# ============================================================
# TAB 4 — Répartition par secteur d'activité
# ============================================================
with tab4:
    st.subheader("🏭 Répartition des offres par secteur d'activité")
    st.caption("Top 20 des secteurs d'activité les plus représentés sur LinkedIn.")

    @st.cache_data
    def get_by_industry():
        query = """
        SELECT
            ci.industry                 AS SECTEUR_ACTIVITE,
            COUNT(DISTINCT jp.job_id)   AS NB_OFFRES
        FROM linkedin.raw_data.job_postings jp
        JOIN linkedin.raw_data.job_industries     ji ON jp.job_id      = ji.job_id
        JOIN linkedin.raw_data.company_industries ci ON ji.industry_id = ci.industry
        GROUP BY ci.industry
        ORDER BY NB_OFFRES DESC
        LIMIT 20
        """
        return session.sql(query).to_pandas()

    df4 = get_by_industry()
    total4 = df4['NB_OFFRES'].sum()
    df4['POURCENTAGE'] = (df4['NB_OFFRES'] / total4 * 100).round(1)

    chart4 = alt.Chart(df4).mark_bar(cornerRadiusTopRight=5).encode(
        x=alt.X('NB_OFFRES:Q', title="Nombre d'offres"),
        y=alt.Y('SECTEUR_ACTIVITE:N', sort='-x', title=""),
        color=alt.Color('NB_OFFRES:Q',
                       scale=alt.Scale(scheme='blues', reverse=False),
                       legend=None),
        tooltip=[
            alt.Tooltip('SECTEUR_ACTIVITE:N', title='Secteur'),
            alt.Tooltip('NB_OFFRES:Q',         title="Nb d'offres"),
            alt.Tooltip('POURCENTAGE:Q',        title='%', format='.1f')
        ]
    ).properties(
        title="Top 20 des secteurs d'activité par nombre d'offres",
        height=550
    )

    text4 = chart4.mark_text(align='left', dx=5, fontSize=11, color='#333').encode(
        text=alt.Text('NB_OFFRES:Q', format=',')
    )

    st.altair_chart(chart4 + text4, use_container_width=True)

    with st.expander("📋 Voir le tableau de données"):
        df4['POURCENTAGE'] = df4['POURCENTAGE'].astype(str) + '%'
        st.dataframe(df4[['SECTEUR_ACTIVITE','NB_OFFRES','POURCENTAGE']], use_container_width=True)

# ============================================================
# TAB 5 — Répartition par type d'emploi
# ============================================================
with tab5:
    st.subheader("⏱️ Répartition des offres par type d'emploi")
    st.caption("Distribution entre Full-time, Part-time, Contract, Internship, etc.")

    @st.cache_data
    def get_by_work_type():
        query = """
        SELECT
            COALESCE(formatted_work_type, 'Non spécifié') AS TYPE_EMPLOI,
            COUNT(*)                                       AS NB_OFFRES,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS POURCENTAGE
        FROM linkedin.raw_data.job_postings
        GROUP BY formatted_work_type
        ORDER BY NB_OFFRES DESC
        """
        return session.sql(query).to_pandas()

    df5 = get_by_work_type()

    linkedin_colors = ['#0077B5', '#00A0DC', '#5BC0EB', '#9BC4E2', '#C8E6F5', '#E8F4FD']

    col_donut5, col_bar5 = st.columns(2)

    with col_donut5:
        chart5_donut = alt.Chart(df5).mark_arc(innerRadius=80, outerRadius=150).encode(
            theta=alt.Theta('NB_OFFRES:Q'),
            color=alt.Color('TYPE_EMPLOI:N',
                           scale=alt.Scale(range=linkedin_colors),
                           legend=alt.Legend(title="Type d'emploi", orient='bottom')),
            tooltip=[
                alt.Tooltip('TYPE_EMPLOI:N',  title='Type'),
                alt.Tooltip('NB_OFFRES:Q',     title="Nb d'offres"),
                alt.Tooltip('POURCENTAGE:Q',   title='%')
            ]
        ).properties(
            title="Répartition par type d'emploi",
            height=400
        )
        st.altair_chart(chart5_donut, use_container_width=True)

    with col_bar5:
        chart5_bar = alt.Chart(df5).mark_bar(cornerRadiusTopRight=5).encode(
            x=alt.X('TYPE_EMPLOI:N', sort='-y', title="Type d'emploi",
                    axis=alt.Axis(labelAngle=-20)),
            y=alt.Y('NB_OFFRES:Q', title="Nombre d'offres"),
            color=alt.Color('TYPE_EMPLOI:N',
                           scale=alt.Scale(range=linkedin_colors),
                           legend=None),
            tooltip=[
                alt.Tooltip('TYPE_EMPLOI:N',  title='Type'),
                alt.Tooltip('NB_OFFRES:Q',     title="Nb d'offres"),
                alt.Tooltip('POURCENTAGE:Q',   title='%')
            ]
        ).properties(
            title="Nombre d'offres par type",
            height=400
        )

        text5 = chart5_bar.mark_text(dy=-8, fontSize=12, fontWeight='bold', color='#333').encode(
            text=alt.Text('NB_OFFRES:Q', format=',')
        )
        st.altair_chart(chart5_bar + text5, use_container_width=True)

    with st.expander("📋 Voir le tableau de données"):
        df5['POURCENTAGE'] = df5['POURCENTAGE'].astype(str) + '%'
        st.dataframe(df5[['TYPE_EMPLOI','NB_OFFRES','POURCENTAGE']], use_container_width=True)

# ============================================================
# FOOTER
# ============================================================
st.markdown("---")
st.markdown(
    """
    <div style='text-align:center; color:#888; font-size:0.85rem;'>
        📊 <strong>LinkedIn Jobs Dashboard</strong> — MBA Big Data & IA — MBAESG 2024/2025<br>
        Données : <em>s3://snowflake-lab-bucket/</em> | Construit avec Snowflake + Streamlit
    </div>
    """,
    unsafe_allow_html=True
)

```


## 11. Conseils pour la rédaction du livrable

Dans ton dépôt GitHub ou ton rendu, tu peux structurer le commentaire de cette manière :

### A. Présentation du contexte
Explique brièvement l'objectif du projet et les données utilisées.

### B. Démarche technique
Décris l'enchaînement logique :
- setup Snowflake,
- création des tables,
- chargement,
- transformation,
- analyses,
- visualisation.

### C. Difficultés rencontrées
Tu peux mentionner par exemple :
- la gestion des JSON ;
- les problèmes potentiels de jointure entre `company_name` et `name` ;
- les valeurs manquantes sur les salaires ;
- les formats de date/timestamp.

### D. Résultats obtenus
Ajoute des captures d'écran du dashboard et commente les tendances observées.


## 12. README du projet

Le fichier `README.md` préparé pour le dépôt est inclus ci-dessous.


```markdown
# 💼 Analyse des Offres d'Emploi LinkedIn avec Snowflake

> **Soumission :** `MBAESG_EVALUATION_ARCHITECTURE_BIGDATA` → axel@logbrain.fr

---

## 📌 Contexte du Projet

Ce projet analyse plusieurs milliers d'offres d'emploi LinkedIn en utilisant **Snowflake** comme entrepôt de données et **Streamlit** pour les visualisations interactives. Les données proviennent d'un bucket **S3 public** et couvrent offres d'emploi, entreprises, secteurs d'activité et compétences.

---

## 👥 Équipe

| Étudiant | Rôle |
|----------|------|
| [Nom Prénom 1] | Setup Snowflake, Scripts SQL (01-03) |
| [Nom Prénom 2] | Analyses SQL (04-05), Application Streamlit |

---

## 🛠️ Stack Technique

| Technologie | Usage |
|-------------|-------|
| **Snowflake** | Entrepôt de données cloud (BDD, Stage, COPY INTO) |
| **SQL** | Manipulation, transformation et analyse |
| **Streamlit** | Dashboard interactif (natif Snowflake) |
| **Altair** | Visualisations (graphiques barres, donut charts) |
| **AWS S3** | Source de données (`s3://snowflake-lab-bucket/`) |

---

## 📁 Structure du Dépôt

```
MBAESG_EVALUATION_ARCHITECTURE_BIGDATA/
│
├── README.md                        ← Ce fichier
│
├── sql/
│   ├── 01_setup_database.sql        ← Création BDD, stage, formats CSV/JSON
│   ├── 02_create_tables.sql         ← Création des 8 tables (CSV + JSON VARIANT)
│   ├── 03_load_data.sql             ← COPY INTO depuis S3
│   ├── 04_transformations.sql       ← JSON brut → tables relationnelles
│   └── 05_analyses.sql              ← 5 requêtes d'analyse avec CTEs
│
├── streamlit/
│   └── streamlit_app.py             ← Dashboard Streamlit complet (5 onglets)
│
└── screenshots/
    ├── analyse_1_top_titres.png
    ├── analyse_2_salaires.png
    ├── analyse_3_taille_entreprise.png
    ├── analyse_4_secteurs.png
    └── analyse_5_types_emploi.png
```

---

## 📊 Jeu de Données (`s3://snowflake-lab-bucket/`)

| Fichier | Format | Description |
|---------|--------|-------------|
| `job_postings.csv` | CSV | Offres d'emploi (27 colonnes) |
| `benefits.csv` | CSV | Avantages par offre |
| `employee_counts.csv` | CSV | Effectifs et followers des entreprises |
| `job_skills.csv` | CSV | Compétences requises par offre |
| `companies.json` | JSON | Détails des entreprises |
| `company_industries.json` | JSON | Secteurs d'activité des entreprises |
| `company_specialities.json` | JSON | Spécialités des entreprises |
| `job_industries.json` | JSON | Secteurs d'activité des offres |

---

## 🚀 Guide de Déploiement

### Prérequis
- Compte Snowflake (trial 120 jours : https://signup.snowflake.com/)
- Connaissances de base en SQL

### Étapes d'exécution

**1. Ouvrir Snowflake** → aller dans `Worksheets`

**2. Exécuter les scripts SQL dans l'ordre :**

```bash
# Ordre obligatoire
01_setup_database.sql    ← Crée la BDD, le stage S3, les formats
02_create_tables.sql     ← Crée les 8 tables
03_load_data.sql         ← Charge les données depuis S3
04_transformations.sql   ← Transforme les JSON en tables relationnelles
05_analyses.sql          ← Exécute les 5 analyses
```

**3. Déployer l'application Streamlit :**
```
Snowflake UI > Projects > Streamlit > + Streamlit App
→ Copier-coller le contenu de streamlit/streamlit_app.py
→ Sélectionner la base linkedin et le schéma raw_data
→ Cliquer sur "Run"
```

---

## 📊 Analyses Réalisées

### Analyse 1 — Top 10 titres les plus publiés par industrie
> **Technique :** CTE + `ROW_NUMBER() OVER (PARTITION BY industry ORDER BY COUNT(*) DESC)`
> Permet de classer les postes au sein de chaque secteur indépendamment.

### Analyse 2 — Top 10 postes les mieux rémunérés par industrie
> **Technique :** CTE + `AVG(med_salary)` + `ROW_NUMBER() OVER (PARTITION BY industry)`
> Seules les offres avec `med_salary IS NOT NULL` sont incluses.

### Analyse 3 — Répartition par taille d'entreprise
> **Technique :** `CASE WHEN` + `LEFT JOIN` sur `companies`
> La colonne `company_size` (0 à 7) est traduite en label lisible.
> Jointure sur `TRIM(LOWER(...))` pour normaliser les noms.

### Analyse 4 — Répartition par secteur d'activité
> **Technique :** Double `JOIN` (job_industries → company_industries)
> + `COUNT(DISTINCT job_id)` pour éviter les doublons, `LIMIT 20`.

### Analyse 5 — Répartition par type d'emploi
> **Technique :** `COALESCE` pour gérer les NULL + `SUM() OVER()` pour le %
> Types : Full-time, Part-time, Contract, Internship, Other.

---

## ⚠️ Problèmes Rencontrés & Solutions

| Problème | Cause | Solution appliquée |
|----------|-------|-------------------|
| Jointure `companies` vide | `company_name` ≠ `name` (espaces/casse) | `TRIM(LOWER(...))` des deux côtés |
| JSON non chargé | Tableau racine `[...]` | `STRIP_OUTER_ARRAY = TRUE` dans le format |
| Colonnes NULL en CSV | Valeurs `"NULL"` littérales | `NULL_IF = ('NULL', 'null', '', 'N/A')` |
| Erreurs de chargement CSV | Caractères spéciaux dans les descriptions | `FIELD_OPTIONALLY_ENCLOSED_BY = '"'` |
| Streamlit session error | App lancée hors Snowflake | Utiliser `get_active_session()` uniquement dans Snowflake |

---

## 📬 Soumission

**Destinataire :** axel@logbrain.fr  
**Objet :** `MBAESG_EVALUATION_ARCHITECTURE_BIGDATA`

---

*MBA Big Data & Intelligence Artificielle — MBAESG — 2024/2025*

```


In [ ]:
# Ce petit bloc Python sert simplement de mémo dans le notebook.
# Il rappelle l'ordre d'exécution recommandé du projet.

etapes = [
    "01_setup_database.sql",
    "02_create_tables.sql",
    "03_load_data.sql",
    "04_transformations.sql",
    "05_analyses.sql",
    "streamlit_app.py"
]

print("Ordre recommandé d'exécution :")
for i, etape in enumerate(etapes, start=1):
    print(f"{i}. {etape}")


## 13. Conclusion

Ce notebook constitue une **version commentée et pédagogique** du projet.

Il peut être utilisé de trois façons :
1. comme **support de compréhension** ;
2. comme **base de livrable** ;
3. comme **guide d'exécution** dans Snowflake.

Si tu veux, je peux maintenant te générer aussi :
- une **version plus académique** du notebook,
- une **version prête pour remise professeur**,
- ou une **version avec davantage d'explications métier et interprétations**.
